In [ ]:
#| default_exp data

# Data

> Cloud Storage, Firestore, Cloud SQL PostgreSQL, Memorystore Redis.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import datetime
import secrets as _sec
import time
from gcpeasy._util import _log, wait_op

try:
    from google.cloud import storage
except ImportError:
    pass
try:
    from google.cloud import firestore as fs
except ImportError:
    pass
try:
    from google.cloud import redis_v1 as redis_client  # noqa: F401
    from google.cloud.redis_v1 import CloudRedisClient, Instance
except ImportError:
    pass
try:
    import googleapiclient.discovery
except ImportError:
    pass

## GCS

In [ ]:
#| export
def _gcs(auth):
    return storage.Client(project=auth.project, credentials=auth.credentials)


def create_bucket(auth, name: str, location: str = None,
                  versioning: bool = True, labels: dict = None,
                  **compliance_opts) -> dict:
    """Create or update a GCS bucket with uniform access, versioning, and encryption.

    Uniform bucket-level access is requested at *creation* time (not patched
    in afterward) so there is no window during which legacy ACLs apply.
    """
    client = _gcs(auth)
    location = location or auth.region
    try:
        bucket = client.get_bucket(name)
        created = False
    except Exception:
        bucket = client.bucket(name)
        bucket.location = location
        # Set uniform access on the local Bucket object before creating
        # so the resource is created with the secure default already on.
        bucket.iam_configuration.uniform_bucket_level_access_enabled = True
        if versioning:
            bucket.versioning_enabled = True
        if labels:
            bucket.labels = labels
        bucket = client.create_bucket(bucket)
        created = True

    if not created:
        changed = False
        if not bucket.iam_configuration.uniform_bucket_level_access_enabled:
            bucket.iam_configuration.uniform_bucket_level_access_enabled = True
            changed = True
        if versioning and not bucket.versioning_enabled:
            bucket.versioning_enabled = True
            changed = True
        if labels and dict(bucket.labels or {}) != labels:
            bucket.labels = {**(bucket.labels or {}), **labels}
            changed = True
        if changed:
            bucket.patch()

    return {'name': bucket.name, 'location': bucket.location,
            'url': f'gs://{bucket.name}'}


def bucket_url(name: str, key: str = '') -> str:
    "Return the gs:// URL for a bucket or object."
    return f'gs://{name}/{key}'.rstrip('/')


def signed_url(auth, name: str, key: str, hours: float = 1,
               method: str = 'GET') -> str:
    """Generate a v4 signed URL for temporary object access.

    Caps expiry at 7 days (the GCP signing v4 hard limit) and rejects
    negative or zero values to avoid accidentally minting expired URLs.
    """
    if hours <= 0:
        raise ValueError('signed_url: hours must be > 0')
    if hours > 24 * 7:
        raise ValueError('signed_url: hours must be <= 168 (7 days, GCP v4 limit)')
    client = _gcs(auth)
    blob = client.bucket(name).blob(key)
    return blob.generate_signed_url(
        version='v4',
        expiration=datetime.timedelta(hours=hours),
        method=method,
        credentials=auth.credentials,
    )


def bucket_conn(name: str) -> str:
    "Return a gs:// connection URI for the bucket."
    return f'gs://{name}'


def delete_bucket(auth, name: str, force: bool = True) -> dict:
    """Delete a GCS bucket.  When ``force=True`` (default), also deletes objects."""
    client = _gcs(auth)
    try:
        bucket = client.get_bucket(name)
    except Exception:
        return {'name': name, 'status': 'not_found'}
    if force:
        for blob in client.list_blobs(bucket):
            blob.delete()
    bucket.delete(force=False)
    return {'name': name, 'status': 'deleted'}

## Firestore

In [ ]:
#| export
def _firestore(auth):
    return fs.Client(project=auth.project, credentials=auth.credentials)


def create_collection(auth, name: str) -> str:
    """Return a Firestore collection name (no-op metadata helper).

    Firestore collections come into existence on first real write — there
    is no API to "create" one.  Earlier versions of this function wrote a
    placeholder ``__init__`` document into every collection, polluting
    application data; that has been removed.

    Use :func:`firestore_conn` to obtain the client and write your first
    real document to materialise the collection.
    """
    return name


def firestore_conn(auth) -> str:
    "Return a firestore:// URI for the project database."
    return f'firestore://{auth.project}/(default)'

## Cloud SQL Postgres (A3)

In [ ]:
#| export
def _sqladmin(auth):
    return googleapiclient.discovery.build(
        'sqladmin', 'v1beta4', credentials=auth.credentials,
        cache_discovery=False,
    )


def _wait_sql_op(sqladmin, project: str, op_name: str, what: str,
                 timeout: int = 1800):
    start = time.monotonic()
    while True:
        op = sqladmin.operations().get(project=project, operation=op_name).execute()
        if op.get('status') == 'DONE':
            if 'error' in op:
                raise RuntimeError(f'{what} failed: {op["error"]}')
            _log(f'{what}: done ({int(time.monotonic() - start)}s)')
            return op
        if time.monotonic() - start > timeout:
            raise TimeoutError(f'{what}: timed out after {timeout}s')
        if int(time.monotonic() - start) % 30 == 0:
            _log(f'{what}: {op.get("status", "PENDING")} '
                 f'({int(time.monotonic() - start)}s)')
        time.sleep(5)


def create_postgres(auth, name: str, tier: str = 'db-g1-small',
                    engine_version: str = 'POSTGRES_16',
                    master_username: str = 'pgadmin',
                    master_password: str = None,
                    deletion_protection: bool = False,
                    backup_retention: int = 7,
                    labels: dict = None,
                    store_password: bool = True,
                    wait: bool = True,
                    **compliance_opts) -> dict:
    """Create or fetch a Cloud SQL PostgreSQL instance.

    * Waits for the instance to exit ``PENDING_CREATE`` when ``wait=True``
      (default) so subsequent calls (DB/user creation) don't race.
    * Auto-generates a strong ``master_password`` when none is given and
      stores it in Secret Manager under
      ``cloudsql/<instance>/master-password`` (when ``store_password=True``,
      default).  The plain-text password is also returned **once** in the
      result on initial creation and never again.
    """
    sqladmin = _sqladmin(auth)
    password = master_password or _sec.token_urlsafe(24)

    body = {
        'name': name,
        'region': auth.region,
        'databaseVersion': engine_version,
        'rootPassword': password,
        'settings': {
            'tier': tier,
            'ipConfiguration': {'requireSsl': True, 'ipv4Enabled': True},
            'backupConfiguration': {
                'enabled': True,
                'transactionLogRetentionDays': min(backup_retention, 7),
                'retainedBackups': backup_retention,
            },
            'deletionProtectionEnabled': deletion_protection,
            'userLabels': labels or {},
        },
    }
    try:
        existing = sqladmin.instances().get(
            project=auth.project, instance=name
        ).execute()
        return {
            'name': name,
            'connection_name': existing.get('connectionName'),
            'state': existing.get('state'),
            'ip_addresses': [a.get('ipAddress') for a in existing.get('ipAddresses', [])],
        }
    except Exception:
        op = sqladmin.instances().insert(
            project=auth.project, body=body
        ).execute()

    if wait:
        _wait_sql_op(sqladmin, auth.project, op['name'],
                     what=f'create_postgres {name}', timeout=1800)
        existing = sqladmin.instances().get(
            project=auth.project, instance=name
        ).execute()
        connection_name = existing.get('connectionName')
        ip_addresses = [a.get('ipAddress') for a in existing.get('ipAddresses', [])]
    else:
        connection_name = f'{auth.project}:{auth.region}:{name}'
        ip_addresses = []

    secret_ref = None
    if store_password and master_password is None:
        try:
            from gcpeasy.network import create_secret
            sec = create_secret(auth, f'cloudsql/{name}/master-password',
                                password, labels=labels or {})
            secret_ref = sec.get('name')
        except Exception as e:  # noqa: BLE001
            _log(f'create_postgres {name}: failed to store password in Secret Manager: {e}')

    return {
        'name': name,
        'connection_name': connection_name,
        'ip_addresses': ip_addresses,
        'master_username': master_username,
        'password': password,  # returned once on initial creation
        'password_secret': secret_ref,
        'operation': op.get('name'),
    }


def postgres_conn(auth, name: str, db: str = 'postgres') -> str:
    "Return a Cloud SQL connection string for use with cloud-sql-python-connector."
    return f'{auth.project}:{auth.region}:{name}'


def create_database(auth, instance: str, name: str,
                    charset: str = 'UTF8', wait: bool = True) -> dict:
    """Create a database inside a Cloud SQL Postgres instance.  Idempotent."""
    sqladmin = _sqladmin(auth)
    try:
        sqladmin.databases().get(
            project=auth.project, instance=instance, database=name,
        ).execute()
        return {'instance': instance, 'database': name, 'status': 'exists'}
    except Exception:
        pass
    body = {'name': name, 'charset': charset, 'instance': instance,
            'project': auth.project}
    op = sqladmin.databases().insert(
        project=auth.project, instance=instance, body=body,
    ).execute()
    if wait:
        _wait_sql_op(sqladmin, auth.project, op['name'],
                     what=f'create_database {name}', timeout=600)
    return {'instance': instance, 'database': name, 'status': 'created'}


def create_db_user(auth, instance: str, username: str,
                   password: str = None, wait: bool = True,
                   store_password: bool = True) -> dict:
    """Create a Cloud SQL Postgres user.

    Auto-generates a password and stores it in Secret Manager under
    ``cloudsql/<instance>/<user>-password`` when ``store_password`` is set.
    """
    sqladmin = _sqladmin(auth)
    pw = password or _sec.token_urlsafe(24)
    try:
        # No GET for users; list and check.
        users = sqladmin.users().list(
            project=auth.project, instance=instance,
        ).execute().get('items', [])
        if any(u.get('name') == username for u in users):
            return {'instance': instance, 'user': username, 'status': 'exists'}
    except Exception:
        pass
    body = {'name': username, 'password': pw, 'instance': instance,
            'project': auth.project}
    op = sqladmin.users().insert(
        project=auth.project, instance=instance, body=body,
    ).execute()
    if wait:
        _wait_sql_op(sqladmin, auth.project, op['name'],
                     what=f'create_db_user {username}', timeout=300)
    secret_ref = None
    if store_password and password is None:
        try:
            from gcpeasy.network import create_secret
            sec = create_secret(auth, f'cloudsql/{instance}/{username}-password', pw)
            secret_ref = sec.get('name')
        except Exception as e:  # noqa: BLE001
            _log(f'create_db_user {username}: failed to store password: {e}')
    return {'instance': instance, 'user': username,
            'password': pw, 'password_secret': secret_ref, 'status': 'created'}


def delete_postgres(auth, name: str, wait: bool = True) -> dict:
    "Delete a Cloud SQL instance.  Idempotent."
    sqladmin = _sqladmin(auth)
    try:
        op = sqladmin.instances().delete(
            project=auth.project, instance=name,
        ).execute()
    except Exception:
        return {'name': name, 'status': 'not_found'}
    if wait:
        _wait_sql_op(sqladmin, auth.project, op['name'],
                     what=f'delete_postgres {name}', timeout=900)
    return {'name': name, 'status': 'deleted'}

## Memorystore Redis

In [ ]:
#| export
def create_redis(auth, name: str, tier: str = 'BASIC', memory_size_gb: int = 1,
                 redis_version: str = 'REDIS_7_0',
                 transit_encryption: bool = True,
                 labels: dict = None,
                 **compliance_opts) -> dict:
    """Create or update a Memorystore Redis instance.

    Reconciles labels on an existing instance when they drift.
    """
    client = CloudRedisClient(credentials=auth.credentials)
    parent = f'projects/{auth.project}/locations/{auth.region}'
    instance_name = f'{parent}/instances/{name}'

    try:
        existing = client.get_instance(name=instance_name)
        if labels:
            wanted = {**dict(existing.labels), **labels}
            if wanted != dict(existing.labels):
                from google.protobuf import field_mask_pb2
                existing.labels.clear()
                existing.labels.update(wanted)
                client.update_instance(
                    instance=existing,
                    update_mask=field_mask_pb2.FieldMask(paths=['labels']),
                )
        return {'name': existing.name, 'host': existing.host,
                'port': existing.port}
    except Exception:
        pass

    instance = Instance(
        display_name=name,
        tier=Instance.Tier[tier],
        memory_size_gb=memory_size_gb,
        redis_version=redis_version,
        transit_encryption_mode=(
            Instance.TransitEncryptionMode.SERVER_AUTHENTICATION
            if transit_encryption
            else Instance.TransitEncryptionMode.DISABLED
        ),
        labels=labels or {},
    )
    op = client.create_instance(parent=parent, instance_id=name, instance=instance)
    result = wait_op(op, what=f'create_redis {name}', timeout=900)
    return {'name': result.name, 'host': result.host, 'port': result.port}


def redis_conn(auth, name: str) -> str:
    "Return a redis:// URI for the Memorystore instance."
    client = CloudRedisClient(credentials=auth.credentials)
    instance = client.get_instance(
        name=f'projects/{auth.project}/locations/{auth.region}/instances/{name}'
    )
    return f'redis://{instance.host}:{instance.port}'


def delete_redis(auth, name: str) -> dict:
    "Delete a Memorystore Redis instance.  Idempotent."
    client = CloudRedisClient(credentials=auth.credentials)
    full = f'projects/{auth.project}/locations/{auth.region}/instances/{name}'
    try:
        op = client.delete_instance(name=full)
    except Exception:
        return {'name': full, 'status': 'not_found'}
    wait_op(op, what=f'delete_redis {name}', timeout=300)
    return {'name': full, 'status': 'deleted'}

### Tests — module exports

In [ ]:
#| hide
import gcpeasy.data as M
for n in ['create_bucket', 'create_collection', 'create_postgres', 'create_redis']:
    assert n in M.__all__